# ForgeGPT: full training run on a Kaggle T4

Trains the ~14M-parameter model (6 layers, 6 heads, width 384, context 256) on all
149M training tokens, then runs four short ablations against a baseline.

**Before running**
1. Settings (right panel): **Accelerator → GPU T4 x2** (one GPU is used), **Internet → On**.
   Internet needs a phone-verified Kaggle account.
2. **Add Input → Datasets → Your Datasets → `forgegpt-tokens-v1`** (the five files from
   `data/kaggle_upload/`).
3. Run it with **Save Version → Save & Run All (Commit)** so it keeps running with the
   browser closed. Everything is saved in the version's **Output** tab.

Expected time: ~45–75 min for the main run, ~30 min for the ablations.

In [ ]:
REPO_URL = "https://github.com/Sahil4504/forgegpt.git"
!git clone -q $REPO_URL /kaggle/working/forgegpt
!pip install -q -e /kaggle/working/forgegpt tabulate
%cd /kaggle/working/forgegpt
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Find the dataset (Kaggle's mount path varies, so search for it)

In [ ]:
import glob, os
meta = glob.glob("/kaggle/input/**/meta.json", recursive=True)
tok = glob.glob("/kaggle/input/**/bpe-8k.json", recursive=True)
assert meta and tok, "Add the forgegpt-tokens-v1 dataset as an input (right panel)"
DATA, TOK = os.path.dirname(meta[0]), tok[0]
print("data:", DATA, sorted(os.listdir(DATA)))
print("tokenizer:", TOK)
OUT = "/kaggle/working/runs"

### Resume (only if a previous session timed out)
Add the earlier version's output as an input; this copies its last checkpoint so training continues.

In [ ]:
import shutil
prev = [p for p in glob.glob("/kaggle/input/**/t4_6L384/ckpt_latest.pt", recursive=True)]
os.makedirs(f"{OUT}/t4_6L384", exist_ok=True)
if prev:
    shutil.copy(prev[0], f"{OUT}/t4_6L384/ckpt_latest.pt")
    print("will resume from", prev[0])
else:
    print("no previous checkpoint: starting fresh")

## Main run

In [ ]:
!python -m forgegpt.train --config configs/kaggle_t4.json --data_dir {DATA} --tokenizer {TOK} --out_dir {OUT}/t4_6L384

## Ablations (2,000 iterations each, one change at a time, same seed and data)

In [ ]:
ablations = {
    "abl_base": "",
    "abl_depth4": "--n_layer 4",
    "abl_depth8": "--n_layer 8",
    "abl_ctx128": "--block_size 128 --batch_size 128",  # same tokens per step
    "abl_lr3e-3": "--lr 0.003 --min_lr 0.0003",
}
common = f"--config configs/kaggle_t4.json --data_dir {DATA} --tokenizer {TOK} --max_iters 2000 --eval_interval 250 --warmup_iters 100"
for name, extra in ablations.items():
    print("=" * 20, name)
    !python -m forgegpt.train {common} --out_dir {OUT}/{name} {extra}

In [ ]:
!python scripts/summarize_runs.py {OUT} --out /kaggle/working/ablations.md

## Loss curves and samples

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(figsize=(8, 4.5))
for path in sorted(glob.glob(f"{OUT}/*/eval.csv")):
    df = pd.read_csv(path)
    ax.plot(df["tokens_seen"] / 1e6, df["val_loss"], label=path.split("/")[-2])
ax.set_xlabel("training tokens (millions)")
ax.set_ylabel("validation loss")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print(open(f"{OUT}/t4_6L384/samples.txt").read()[-2500:])

## Package the results
Writes `/kaggle/working/forgegpt_results/`: a weights-only checkpoint (~55 MB instead of
~170 MB with optimizer state), the logs of every run, and the ablation table. Download that
folder from the **Output** tab and put it in `E:\Project\GPT2\runs\kaggle\`.

In [ ]:
import json
import torch

res = "/kaggle/working/forgegpt_results"
os.makedirs(res, exist_ok=True)
ckpt = torch.load(f"{OUT}/t4_6L384/ckpt_best.pt", map_location="cpu", weights_only=False)
slim = {k: ckpt[k] for k in ("model", "model_config", "iter", "best_val", "tokens_seen")}
torch.save(slim, f"{res}/ckpt_best.pt")
for run in sorted(os.listdir(OUT)):
    os.makedirs(f"{res}/logs/{run}", exist_ok=True)
    for f in ("eval.csv", "train_log.csv", "config.json", "summary.json", "samples.txt"):
        if os.path.exists(f"{OUT}/{run}/{f}"):
            shutil.copy(f"{OUT}/{run}/{f}", f"{res}/logs/{run}/{f}")
shutil.copy("/kaggle/working/ablations.md", f"{res}/ablations.md")
info = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        **json.load(open(f"{OUT}/t4_6L384/summary.json"))}
json.dump(info, open(f"{res}/run_info.json", "w"), indent=2)
print(json.dumps(info, indent=2))
!du -sh {res}